In [1]:
import pandas as pd
import numpy as np
import joblib

print("Final pipeline libraries imported successfully.")

Final pipeline libraries imported successfully.


In [2]:
DATA_PATH = "../data/processed/ml_dataset.csv"
MODEL_PATH = "../models/improved_random_forest_glucose_model.pkl"
FEATURE_PATH = "../models/feature_columns.pkl"

df = pd.read_csv(DATA_PATH)

df["timestamp"] = pd.to_datetime(df["timestamp"])

model = joblib.load(MODEL_PATH)
feature_columns = joblib.load(FEATURE_PATH)

print("Dataset loaded:", df.shape)
print("Model loaded successfully.")
print("Expected features:", len(feature_columns))

Dataset loaded: (65499, 21)
Model loaded successfully.
Expected features: 18


In [3]:
# Select a patient record with lifestyle information

candidates = df[
    (df["meal_carbs"] > 0) |
    (df["exercise_duration"] > 0) |
    (df["sleep_duration_hours"] > 0)
].copy()

sample = candidates.sort_values("timestamp").iloc[-1].copy()

print("PATIENT INPUT")
print("-----------------------------------")
print("Patient:", sample["Patient"])
print("Timestamp:", sample["timestamp"])
print("Current glucose:", sample["glucose_value"], "mg/dL")
print("Meal carbs:", sample["meal_carbs"], "g")
print("Bolus dose:", sample["bolus_dose"])
print("Exercise:", sample["exercise_duration"], "minutes")
print("Sleep:", sample["sleep_duration_hours"], "hours")
print("Sleep quality:", sample["sleep_quality"])

PATIENT INPUT
-----------------------------------
Patient: 540-ws-training
Timestamp: 2027-07-03 23:26:44
Current glucose: 267 mg/dL
Meal carbs: 100.0 g
Bolus dose: 0.9
Exercise: 0.0 minutes
Sleep: 0.0 hours
Sleep quality: 0.0


In [4]:
def predict_glucose(state):
    X = pd.DataFrame([state])
    X = X[feature_columns]

    return model.predict(X)[0]


baseline_prediction = predict_glucose(sample)

print(
    "30-minute predicted glucose:",
    round(baseline_prediction, 2),
    "mg/dL"
)

30-minute predicted glucose: 266.89 mg/dL


In [5]:
def glucose_status(glucose):

    if glucose < 70:
        return "LOW"

    elif glucose < 140:
        return "NORMAL"

    elif glucose < 180:
        return "ELEVATED"

    else:
        return "HIGH"


status = glucose_status(baseline_prediction)

print("Predicted glucose status:", status)

Predicted glucose status: HIGH


In [6]:
observations = []
recommendations = []

if sample["meal_carbs"] >= 50:

    observations.append(
        "High carbohydrate intake detected."
    )

    recommendations.append(
        "Consider a lower-carbohydrate meal option."
    )


if sample["exercise_duration"] == 0:

    observations.append(
        "No exercise recorded."
    )

    recommendations.append(
        "Consider maintaining appropriate physical activity."
    )


if sample["sleep_duration_hours"] < 6:

    observations.append(
        "Sleep duration is relatively short."
    )

    recommendations.append(
        "Consider maintaining adequate sleep."
    )


if sample["sleep_quality"] <= 2:

    observations.append(
        "Sleep quality is relatively low."
    )

    recommendations.append(
        "Consider improving sleep quality and maintaining a consistent sleep routine."
    )


if sample["glucose_change_15min"] >= 15:

    observations.append(
        "Glucose is rising rapidly."
    )


elif sample["glucose_change_15min"] <= -15:

    observations.append(
        "Glucose is falling rapidly."
    )


if not observations:

    observations.append(
        "No major lifestyle concerns detected."
    )


print("LIFESTYLE ANALYSIS")
print("-----------------------------------")

for item in observations:
    print("•", item)

LIFESTYLE ANALYSIS
-----------------------------------
• High carbohydrate intake detected.
• No exercise recorded.
• Sleep duration is relatively short.
• Sleep quality is relatively low.
• Glucose is rising rapidly.


In [7]:
current = sample.copy()

reduced_carbs = sample.copy()
reduced_carbs["meal_carbs"] = max(
    0,
    sample["meal_carbs"] * 0.5
)

increased_exercise = sample.copy()

if sample["exercise_duration"] > 0:

    increased_exercise["exercise_duration"] = (
        sample["exercise_duration"] + 20
    )

else:

    increased_exercise["exercise_duration"] = 20

increased_exercise["exercise_intensity"] = max(
    sample["exercise_intensity"],
    1
)

improved_sleep = sample.copy()

improved_sleep["sleep_duration_hours"] = max(
    sample["sleep_duration_hours"],
    8
)

improved_sleep["sleep_quality"] = max(
    sample["sleep_quality"],
    4
)


scenarios = {

    "Current Lifestyle": current,

    "Reduced Carbohydrates": reduced_carbs,

    "Increased Exercise": increased_exercise,

    "Improved Sleep": improved_sleep
}


scenario_results = []

for name, state in scenarios.items():

    prediction = predict_glucose(state)

    scenario_results.append({

        "Scenario": name,

        "Predicted_Glucose_30min": prediction
    })


scenario_results = pd.DataFrame(scenario_results)

print("DIGITAL TWIN SIMULATION")
print("-----------------------------------")

display(
    scenario_results.round(2)
)

DIGITAL TWIN SIMULATION
-----------------------------------


,Scenario,Predicted_Glucose_30min
0,Current Lifestyle,266.89
1,Reduced Carbohydrates,263.93
2,Increased Exercise,266.89
3,Improved Sleep,269.64


In [8]:
baseline = scenario_results.loc[
    scenario_results["Scenario"] == "Current Lifestyle",
    "Predicted_Glucose_30min"
].iloc[0]


alternative_results = scenario_results[
    scenario_results["Scenario"] != "Current Lifestyle"
].copy()


alternative_results["Improvement"] = (
    baseline -
    alternative_results["Predicted_Glucose_30min"]
)


MEANINGFUL_IMPROVEMENT = 2.0


meaningful = alternative_results[
    alternative_results["Improvement"] >=
    MEANINGFUL_IMPROVEMENT
]


if len(meaningful) > 0:

    best = meaningful.loc[
        meaningful["Improvement"].idxmax()
    ]

    best_scenario = best["Scenario"]

    best_prediction = best[
        "Predicted_Glucose_30min"
    ]

    improvement = best["Improvement"]

    decision = (
        f"Consider the {best_scenario.lower()} "
        "scenario as a simulated lifestyle option."
    )

else:

    best_scenario = "Current Lifestyle"

    best_prediction = baseline

    improvement = 0

    decision = (
        "No scenario produced a meaningful simulated improvement. "
        "Maintain current lifestyle state."
    )


print("DECISION ENGINE")
print("-----------------------------------")

print("Baseline:", round(baseline, 2), "mg/dL")

print("Best scenario:", best_scenario)

print(
    "Improvement:",
    round(improvement, 2),
    "mg/dL"
)

print("Decision:", decision)

DECISION ENGINE
-----------------------------------
Baseline: 266.89 mg/dL
Best scenario: Reduced Carbohydrates
Improvement: 2.96 mg/dL
Decision: Consider the reduced carbohydrates scenario as a simulated lifestyle option.


In [9]:
print("=" * 60)
print("          DIA DECISION AI")
print("      PERSONALIZED DECISION SUPPORT")
print("=" * 60)

print()

print("PATIENT")
print("-----------------------------------")
print("Patient:", sample["Patient"])
print("Timestamp:", sample["timestamp"])

print()

print("GLUCOSE PREDICTION")
print("-----------------------------------")
print(
    "Current glucose:",
    sample["glucose_value"],
    "mg/dL"
)

print(
    "Predicted glucose (30 min):",
    round(baseline_prediction, 2),
    "mg/dL"
)

print("Status:", status)

print()

print("LIFESTYLE ANALYSIS")
print("-----------------------------------")

for item in observations:
    print("•", item)

print()

print("RECOMMENDATIONS")
print("-----------------------------------")

for item in recommendations:
    print("•", item)

print()

print("DIGITAL TWIN DECISION")
print("-----------------------------------")

print(
    "Best simulated scenario:",
    best_scenario
)

print(
    "Simulated glucose:",
    round(best_prediction, 2),
    "mg/dL"
)

print(
    "Simulated improvement:",
    round(improvement, 2),
    "mg/dL"
)

print()

print("FINAL DECISION")
print("-----------------------------------")
print(decision)

print()

print("=" * 60)
print("Pipeline completed successfully.")
print("=" * 60)

          DIA DECISION AI
      PERSONALIZED DECISION SUPPORT

PATIENT
-----------------------------------
Patient: 540-ws-training
Timestamp: 2027-07-03 23:26:44

GLUCOSE PREDICTION
-----------------------------------
Current glucose: 267 mg/dL
Predicted glucose (30 min): 266.89 mg/dL
Status: HIGH

LIFESTYLE ANALYSIS
-----------------------------------
• High carbohydrate intake detected.
• No exercise recorded.
• Sleep duration is relatively short.
• Sleep quality is relatively low.
• Glucose is rising rapidly.

RECOMMENDATIONS
-----------------------------------
• Consider a lower-carbohydrate meal option.
• Consider maintaining appropriate physical activity.
• Consider maintaining adequate sleep.
• Consider improving sleep quality and maintaining a consistent sleep routine.

DIGITAL TWIN DECISION
-----------------------------------
Best simulated scenario: Reduced Carbohydrates
Simulated glucose: 263.93 mg/dL
Simulated improvement: 2.96 mg/dL

FINAL DECISION
-----------------------

In [10]:
final_result = {

    "patient": sample["Patient"],

    "timestamp": str(sample["timestamp"]),

    "current_glucose": float(
        sample["glucose_value"]
    ),

    "predicted_glucose_30min": float(
        baseline_prediction
    ),

    "glucose_status": status,

    "best_scenario": best_scenario,

    "best_scenario_prediction": float(
        best_prediction
    ),

    "simulated_improvement": float(
        improvement
    ),

    "decision": decision
}


result_df = pd.DataFrame([final_result])

OUTPUT_PATH = "../data/processed/final_pipeline_result.csv"

result_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Final pipeline result saved to:")
print(OUTPUT_PATH)

display(result_df)

Final pipeline result saved to:
../data/processed/final_pipeline_result.csv


,patient,timestamp,current_glucose,predicted_glucose_30min,glucose_status,best_scenario,best_scenario_prediction,simulated_improvement,decision
0,540-ws-training,2027-07-03 23:26:44,267.0,266.889672,HIGH,Reduced Carbohydrates,263.928316,2.961356,Consider the reduced carbohydrates scenario as...
